<a href="https://colab.research.google.com/github/plwam/hugging_face_colab/blob/main/YOLOv8_TACO_finetuning_first_try.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Cellule 1 — Installer les dépendances
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00


In [2]:
#Cellule 2 — Vérifier le GPU
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")
# """ Assure-toi avant de lancer cette cellule que tu as bien activé
# le GPU dans Colab : Runtime (Exécution) → Change runtime
# type → Hardware accelerator → GPU (T4 gratuit)."""

GPU disponible : True
Nom du GPU : Tesla T4


' Assure-toi avant de lancer cette cellule que tu as bien activé\nle GPU dans Colab : Runtime (Exécution) → Change runtime \ntype → Hardware accelerator → GPU (T4 gratuit).'

In [3]:
# """ Cellule 3 — Télécharger le dataset TACO depuis Roboflow
# Colle ton snippet ici (remplace par ta vraie clé API) :"""


from roboflow import Roboflow
rf = Roboflow(api_key="cNaM3djoflVzcir9njEy")
project = rf.workspace("paterne-lwanzo").project("taco-trash-annotations-in-context-ucsi6")
version = project.version(1)
dataset = version.download("yolov8")

# """ Ça va créer un dossier (le nom exact s'affichera après exécution,
# généralement un truc comme /content/taco-trash-annotations-in-context-ucsi6-1)."""

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to TACO:-Trash-Annotations-in-Context-Dataset-1 in yolov8:: 100%|██████████| 3003/3003 [00:01<00:00, 2201.33it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


" Ça va créer un dossier (le nom exact s'affichera après exécution, \ngénéralement un truc comme /content/taco-trash-annotations-in-context-ucsi6-1)."

In [5]:
# Cellule 4 — Explorer la structure téléchargée

import os
dataset_path = dataset.location
print(dataset_path)
os.listdir(dataset_path)



/content/TACO:-Trash-Annotations-in-Context-Dataset-1


['train',
 'valid',
 'test',
 'data.yaml',
 'README.dataset.txt',
 'README.roboflow.txt']

In [6]:
# On passe maintenant au script de remapping des classes.
# D'abord, vérifions ensemble le contenu exact de data.yaml pour connaître l'ordre exact des 59 classes (essentiel pour faire un mapping correct)

with open(f"{dataset_path}/data.yaml", "r") as f:
    print(f.read())

names:
- Aerosol
- Aluminium blister pack
- Aluminium foil
- Battery
- Broken glass
- Carded blister pack
- Cigarette
- Clear plastic bottle
- Corrugated carton
- Crisp packet
- Disposable food container
- Disposable plastic cup
- Drink can
- Drink carton
- Egg carton
- Foam cup
- Foam food container
- Food Can
- Food waste
- Garbage bag
- Glass bottle
- Glass cup
- Glass jar
- Magazine paper
- Meal carton
- Metal bottle cap
- Metal lid
- Normal paper
- Other carton
- Other plastic
- Other plastic bottle
- Other plastic container
- Other plastic cup
- Other plastic wrapper
- Paper bag
- Paper cup
- Paper straw
- Pizza box
- Plastic bottle cap
- Plastic film
- Plastic glooves
- Plastic lid
- Plastic straw
- Plastic utensils
- Polypropylene bag
- Pop tab
- Rope - strings
- Scrap metal
- Shoe
- Single-use carrier bag
- Six pack rings
- Spread tub
- Squeezable tube
- Styrofoam piece
- Tissues
- Toilet tube
- Tupperware
- Unlabeled litter
- Wrapping paper
nc: 59
roboflow:
  license: CC BY 4

In [7]:
# Voici le script complet à mettre en Cellule 5 :

import os
import yaml
import shutil

# --- 1. Charger les classes d'origine ---
with open(f"{dataset_path}/data.yaml", "r") as f:
    data_cfg = yaml.safe_load(f)

original_classes = data_cfg["names"]
print(f"{len(original_classes)} classes d'origine chargées.")

# --- 2. Définir le mapping ancien nom -> nouvelle catégorie ---
mapping = {
    "Aerosol": "metal", "Aluminium blister pack": "metal", "Aluminium foil": "metal",
    "Drink can": "metal", "Food Can": "metal", "Metal bottle cap": "metal",
    "Metal lid": "metal", "Pop tab": "metal", "Scrap metal": "metal",

    "Broken glass": "glass", "Glass bottle": "glass", "Glass cup": "glass", "Glass jar": "glass",

    "Corrugated carton": "carton", "Drink carton": "carton", "Egg carton": "carton",
    "Magazine paper": "carton", "Meal carton": "carton", "Normal paper": "carton",
    "Other carton": "carton", "Paper bag": "carton", "Paper cup": "carton",
    "Paper straw": "carton", "Pizza box": "carton", "Tissues": "carton",
    "Toilet tube": "carton", "Wrapping paper": "carton",

    "Clear plastic bottle": "plastic", "Crisp packet": "plastic",
    "Disposable food container": "plastic", "Disposable plastic cup": "plastic",
    "Foam cup": "plastic", "Foam food container": "plastic", "Garbage bag": "plastic",
    "Other plastic": "plastic", "Other plastic bottle": "plastic",
    "Other plastic container": "plastic", "Other plastic cup": "plastic",
    "Other plastic wrapper": "plastic", "Plastic bottle cap": "plastic",
    "Plastic film": "plastic", "Plastic glooves": "plastic", "Plastic lid": "plastic",
    "Plastic straw": "plastic", "Plastic utensils": "plastic",
    "Polypropylene bag": "plastic", "Single-use carrier bag": "plastic",
    "Six pack rings": "plastic", "Spread tub": "plastic", "Squeezable tube": "plastic",
    "Styrofoam piece": "plastic", "Tupperware": "plastic",

    "Battery": "other", "Carded blister pack": "other", "Cigarette": "other",
    "Food waste": "other", "Rope - strings": "other", "Shoe": "other",

    "Unlabeled litter": None,  # exclue
}

new_classes = ["plastic", "carton", "glass", "metal", "other"]
new_class_to_id = {name: i for i, name in enumerate(new_classes)}

# old_id -> new_id (ou None si à exclure)
old_id_to_new_id = {}
for old_id, old_name in enumerate(original_classes):
    target = mapping.get(old_name)
    old_id_to_new_id[old_id] = new_class_to_id[target] if target else None

# --- 3. Fonction pour remapper un dossier labels/ ---
def remap_labels(labels_dir):
    files = [f for f in os.listdir(labels_dir) if f.endswith(".txt")]
    total_lines, kept_lines = 0, 0
    for fname in files:
        path = os.path.join(labels_dir, fname)
        new_lines = []
        with open(path, "r") as f:
            for line in f:
                total_lines += 1
                parts = line.strip().split()
                if not parts:
                    continue
                old_id = int(parts[0])
                new_id = old_id_to_new_id.get(old_id)
                if new_id is not None:
                    new_lines.append(" ".join([str(new_id)] + parts[1:]))
                    kept_lines += 1
        with open(path, "w") as f:
            f.write("\n".join(new_lines) + ("\n" if new_lines else ""))
    print(f"{labels_dir} : {kept_lines}/{total_lines} annotations conservées")

# --- 4. Appliquer sur train/valid/test ---
for split in ["train", "valid", "test"]:
    remap_labels(f"{dataset_path}/{split}/labels")

# --- 5. Réécrire data.yaml ---
data_cfg["names"] = new_classes
data_cfg["nc"] = len(new_classes)
with open(f"{dataset_path}/data.yaml", "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print("\nRemapping terminé. Nouvelles classes :", new_classes)

59 classes d'origine chargées.
/content/TACO:-Trash-Annotations-in-Context-Dataset-1/train/labels : 2925/3250 annotations conservées
/content/TACO:-Trash-Annotations-in-Context-Dataset-1/valid/labels : 917/1032 annotations conservées
/content/TACO:-Trash-Annotations-in-Context-Dataset-1/test/labels : 496/577 annotations conservées

Remapping terminé. Nouvelles classes : ['plastic', 'carton', 'glass', 'metal', 'other']
